# SAM 3 + YOLO11m-seg + Active Learning (Kaggle GPU Setup)

**Bật sẵn trước khi chạy trên Kaggle:**
1. **Accelerator:** `GPU T4 x2` hoặc `GPU P100` (16 GB VRAM)
2. **Internet:** `ON`
3. **Add-ons -> Secrets (tùy chọn để tải SAM 3 tự động):** thêm secret tên `HF_TOKEN` (token Hugging Face quyền Read đã được duyệt `facebook/sam3`).

In [ ]:
!nvidia-smi
!free -h
%cd /kaggle/working
!rm -rf hackathon_vinai
!git clone https://github.com/bayvai20kg/hackathon_vinai.git
%cd /kaggle/working/hackathon_vinai

In [ ]:
!pip install -q "ultralytics>=8.4.210" cvat-sdk huggingface_hub psutil PyYAML
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Bước 1: Tải `sam3.pt` từ Hugging Face (`facebook/sam3`)
- Nếu đã lưu `HF_TOKEN` trong Kaggle Secrets, cell dưới sẽ tự lấy.
- Nếu chưa lưu secret, thay `""` bằng token `hf_...` của bạn (hoặc bỏ qua nếu muốn chạy `sam2.1_b.pt` fallback).

In [ ]:
import os
from huggingface_hub import hf_hub_download

hf_token = ""  # Hoặc dán token hf_... vào đây nếu không dùng Kaggle Secrets
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    path = hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_dir="weights", token=hf_token)
    print("Downloaded SAM 3 to:", path)
else:
    print("Chưa có HF_TOKEN -> pipeline sẽ tự dùng SAM 2.1 fallback (sam2.1_b.pt).")

## Bước 2A (Tùy chọn): Chạy thử 50 ảnh với `YOLO11m-seg` + `SAM 3` trên Kaggle

In [ ]:
!python run_pipeline.py run-demo-50 --config configs/pipeline_kaggle.yaml --num-images 50 --epochs 10

## Bước 2B: Vòng Lặp Warm-up thật (200 ảnh -> CVAT -> Train YOLO11m-seg)
1. Chạy `warmup-export` tạo `pipeline_work/cvat_export/warmup_batch/warmup_batch_coco.zip` + `cvat_labels.json`.
2. Tải file ZIP đó ở tab **Output** bên phải Kaggle, đưa lên CVAT sửa nhãn 11 lớp.
3. Export lại file `.zip` từ CVAT, upload lên Kaggle (dạng Kaggle Dataset hoặc `/kaggle/input/...`), rồi chạy cell `import-cvat` + `train` bên dưới.

In [ ]:
!python run_pipeline.py warmup-export --config configs/pipeline_kaggle.yaml --num-images 200 --batch-name warmup_batch

In [ ]:
# Sau khi đã sửa trên CVAT và upload file zip đã sửa vào /kaggle/input/...
CVAT_REVIEWED_ZIP = "/kaggle/input/your-cvat-dataset/warmup_reviewed.zip"

!python run_pipeline.py import-cvat --config configs/pipeline_kaggle.yaml --source {CVAT_REVIEWED_ZIP}
!python run_pipeline.py train --config configs/pipeline_kaggle.yaml --run-name round_01 --epochs 40

## Bước 3: Chạy Active Learning + SAM 3 Auto-labeling (`>0.85`) + Export ảnh khó (`<0.60`)

In [ ]:
!python run_pipeline.py al-step --config configs/pipeline_kaggle.yaml --weights pipeline_work/runs/round_01/weights/best.pt --batch-name al_round_01

In [ ]:
# Sau khi sửa al_round_01_coco.zip trên CVAT và upload lên Kaggle:
AL_REVIEWED_ZIP = "/kaggle/input/your-cvat-dataset/al_round_01_reviewed.zip"

!python run_pipeline.py import-cvat --config configs/pipeline_kaggle.yaml --source {AL_REVIEWED_ZIP}
!python run_pipeline.py train --config configs/pipeline_kaggle.yaml --run-name round_02 --weights pipeline_work/runs/round_01/weights/best.pt --epochs 35